In [0]:
import requests
import json

print("Preparando a requisição para a API do OpenWeatherMap... ")

# 1. Configurações da API
API_KEY = "a81de9c069e7d5c36deba73532ad32e9"
CIDADE = "Toronto"
URL = f"http://api.openweathermap.org/data/2.5/weather?q={CIDADE}&appid={API_KEY}&units=metric"

# 2. Fazendo o Pedido (Requisição GET)
resposta = requests.get(URL)

# 3. Verificando se deu certo (O código 200 significa Sucesso / OK)
if resposta.status_code == 200:
    print(f"Sucesso! Os dados de {CIDADE} chegaram perfeitamente.\n")
    # Transformando a resposta em um Dicionário Python (JSON)
    dados_clima = resposta.json()
    # Imprimindo a estrutura crua para entendermos o que veio
    # Usamos json.dumps para imprimir formatado
    print(json.dumps(dados_clima, indent=4))
else:
    print(f"Erro ao acessar a API! Codigo de Status: {resposta.status_code}")
    print(resposta.text)


Preparando a requisição para a API do OpenWeatherMap... 
Sucesso! Os dados de Toronto chegaram perfeitamente.

{
    "coord": {
        "lon": -79.4163,
        "lat": 43.7001
    },
    "weather": [
        {
            "id": 800,
            "main": "Clear",
            "description": "clear sky",
            "icon": "01d"
        }
    ],
    "base": "stations",
    "main": {
        "temp": -12.97,
        "feels_like": -19.97,
        "temp_min": -13.94,
        "temp_max": -12.18,
        "pressure": 1039,
        "humidity": 77,
        "sea_level": 1039,
        "grnd_level": 1022
    },
    "visibility": 10000,
    "wind": {
        "speed": 5.81,
        "deg": 10,
        "gust": 0
    },
    "clouds": {
        "all": 0
    },
    "dt": 1772456622,
    "sys": {
        "type": 2,
        "id": 2099289,
        "country": "CA",
        "sunrise": 1772452371,
        "sunset": 1772492826
    },
    "timezone": -18000,
    "id": 6167865,
    "name": "Toronto",
    "cod": 200


## CAMADA BRONZE 🥉
- 1. Escalar: fazer um loop
- 2. Estruturar: Pegar os dados e colocar no Pyspark tranformando em um DataFrame
- 3. Armazenar: Salvar isso na nossa Camada Bronze em formato Delta (usando o modo .mode("append")

In [0]:
import requests
from pyspark.sql.functions import col, from_json, schema_of_json, lit

print("Iniciando a Ingestão de Dados (Camada Bronze)...")

# 1. Configurações
API_KEY = "a81de9c069e7d5c36deba73532ad32e9" 
cidades_alvo = ["Toronto", "Vancouver", "Montreal", "Calgary", "Ottawa"]

# Desta vez, vamos guardar o TEXTO puro, e não o dicionário
dados_brutos_texto = [] 

# 2. Extração em Massa
for cidade in cidades_alvo:
    url = f"http://api.openweathermap.org/data/2.5/weather?q={cidade}&appid={API_KEY}&units=metric"
    resposta = requests.get(url)
    
    if resposta.status_code == 200:
        # Pegamos a resposta em formato de TEXTO (.text em vez de .json())
        dados_brutos_texto.append(resposta.text)
        print(f"✅ Dados extraídos: {cidade}")
    else:
        print(f"❌ Falha ao extrair {cidade}: {resposta.status_code}")

print("\nAplicando Magia Serverless (Processamento 100% em Memória)...")

# 3. Criando uma Tabela com o texto bruto
# O Spark exige uma lista de listas para criar a tabela: [["{json1}"], ["{json2}"]]
lista_para_spark = [[texto] for texto in dados_brutos_texto]
df_textos = spark.createDataFrame(lista_para_spark, schema=["json_cru"])

# 4. Ensinando o Spark a ler o JSON sozinho
# Pegamos o primeiro JSON da lista e pedimos pro Spark descobrir o esquema (a estrutura)
esquema_descoberto = schema_of_json(lit(dados_brutos_texto[0]))

# Aplicamos esse esquema na nossa tabela para "desempacotar" tudo e expandir as colunas (.*)
df_bronze = df_textos.withColumn("dados_estruturados", from_json(col("json_cru"), esquema_descoberto)) \
                     .select("dados_estruturados.*")

# 5. Salvando na Camada Bronze 
df_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("bronze_weather")

print("Camada Bronze atualizada com sucesso! 🚀🥉")

# Exibindo o resultado da nossa tabela final
display(spark.table("bronze_weather"))

Iniciando a Ingestão de Dados (Camada Bronze)...
✅ Dados extraídos: Toronto
✅ Dados extraídos: Vancouver
✅ Dados extraídos: Montreal
✅ Dados extraídos: Calgary
✅ Dados extraídos: Ottawa

Aplicando Magia Serverless (Processamento 100% em Memória)...
Camada Bronze atualizada com sucesso! 🚀🥉


base,clouds,cod,coord,dt,id,main,name,sys,timezone,visibility,weather,wind
stations,List(0),200,"List(43.7001, -79.4163)",1772459519,6167865,"List(-17.6, 1022, 65, 1039, 1039, -10.6, -9.17, -11.56)",Toronto,"List(CA, 2099289, 1772452371, 1772492826, 2)",-18000,10000,"List(List(clear sky, 01d, 800, Clear))","List(10, 0, 5.81)"
stations,List(71),200,"List(49.2497, -123.1193)",1772459224,6173331,"List(2.99, 1010, 94, 1018, 1018, 2.99, 5.57, 1.64)",Vancouver,"List(CA, 2011597, 1772463171, 1772503000, 2)",-28800,10000,"List(List(broken clouds, 04n, 803, Clouds))","List(3, null, 0.89)"
stations,List(20),200,"List(45.5088, -73.5878)",1772459694,6077243,"List(-17.18, 1036, 68, 1040, 1040, -17.18, -15.88, -19.3)",Montreal,"List(CA, 2095416, 1772451073, 1772491326, 2)",-18000,10000,"List(List(few clouds, 02d, 801, Clouds))","List(260, null, 1.03)"
stations,List(20),200,"List(51.0501, -114.0853)",1772459530,5913490,"List(-10.34, 884, 70, 1016, 1016, -4.85, -3.77, -5.83)",Calgary,"List(CA, 989, 1772461126, 1772500709, 1)",-25200,10000,"List(List(few clouds, 02n, 801, Clouds))","List(50, null, 4.12)"
stations,List(17),200,"List(45.4112, -75.6981)",1772459222,6094817,"List(-20.17, 1028, 67, 1041, 1041, -16.42, -14.68, -18.47)",Ottawa,"List(CA, 2005537, 1772451573, 1772491839, 2)",-18000,10000,"List(List(few clouds, 02d, 801, Clouds))","List(226, null, 1.38)"


## CAMADA SILVER 🥈
- 1. Desempacotar
- 2. Traduzir as colunas para nomes
- 3. Transformar a coluna `dt` numa data e hora


In [0]:
%sql
-- Criando ou substituindo a tabela Silver para garantir que ela sempre fique atualizada
CREATE OR REPLACE TABLE silver_weather AS
SELECT
-- 1. Identificação
name as city_name,
sys.country AS country_code,

-- 2. Desempacotando a coluna 'main' (Temperaturas)
main.temp AS temperature_c,
main.feels_like AS feels_like_c,
main.humidity AS humidity_percent,

-- 3. Desempacotando o clima (A coluna 'weather' é uma lista, então pegamos o item [0])
weather[0].main AS weather_condition,
weather[0].description AS weather_description,

-- 4. Desempacotando o vento
wind.speed AS wind_speed_m_s,

-- 5. Tratamento de Data: O 'dt' vem em formato Unix segundos desde 1970. 
-- A função timestamp converte isso magicamente para Data e Hora reais!
timestamp(dt) AS extraction_timestamp

FROM bronze_weather;

-- Exibindo o resultado final para conferência
SELECT * FROM silver_weather;

city_name,country_code,temperature_c,feels_like_c,humidity_percent,weather_condition,weather_description,wind_speed_m_s,extraction_timestamp
Toronto,CA,-10.6,-17.6,65,Clear,clear sky,5.81,2026-03-02T13:51:59.000Z
Vancouver,CA,2.99,2.99,94,Clouds,broken clouds,0.89,2026-03-02T13:47:04.000Z
Montreal,CA,-17.18,-17.18,68,Clouds,few clouds,1.03,2026-03-02T13:54:54.000Z
Calgary,CA,-4.85,-10.34,70,Clouds,few clouds,4.12,2026-03-02T13:52:10.000Z
Ottawa,CA,-16.42,-20.17,67,Clouds,few clouds,1.38,2026-03-02T13:47:02.000Z


## CAMADA GOLD 🥇 
- Agrupar os dados por cidade (PARTITION BY city_name).

- Ordenar do mais recente para o mais antigo (ORDER BY extraction_timestamp DESC).

- Dar o número "1" para a linha mais nova de cada cidade.

- Filtrar e salvar apenas as linhas com o número 1.

In [0]:
%sql
-- Criando a Camada Gold com a "Foto Atual" do Clima
CREATE OR REPLACE TABLE gold_latest_weather AS

-- 1. Usamos uma CTE (Common Table Expression - a cláusula WITH) para organizar a lógica
WITH RankedWeather AS (
    SELECT
        city_name,
        country_code,
        temperature_c,
        feels_like_c,
        humidity_percent,
        weather_condition,
        weather_description,
        wind_speed_m_s,
        extraction_timestamp,
        -- Aqui está a mágica: Numerando as linhas por cidade, da mais nova para a mais velha
        ROW_NUMBER() OVER(PARTITION BY city_name ORDER BY extraction_timestamp DESC) as rn
    FROM silver_weather
)
-- 2. Selecionamos tudo da CTE, filtrando APENAS a linha 1 (a leitura mais recente de cada cidade)
SELECT 
    city_name,
    country_code,
    temperature_c,
    feels_like_c,
    humidity_percent,
    weather_condition,
    weather_description,
    wind_speed_m_s,
    extraction_timestamp
FROM RankedWeather
WHERE rn = 1;

-- Conferindo a tabela final que vai alimentar o nosso Dashboard
SELECT * FROM gold_latest_weather;

city_name,country_code,temperature_c,feels_like_c,humidity_percent,weather_condition,weather_description,wind_speed_m_s,extraction_timestamp
Calgary,CA,-4.85,-10.34,70,Clouds,few clouds,4.12,2026-03-02T13:52:10.000Z
Montreal,CA,-17.18,-17.18,68,Clouds,few clouds,1.03,2026-03-02T13:54:54.000Z
Ottawa,CA,-16.42,-20.17,67,Clouds,few clouds,1.38,2026-03-02T13:47:02.000Z
Toronto,CA,-10.6,-17.6,65,Clear,clear sky,5.81,2026-03-02T13:51:59.000Z
Vancouver,CA,2.99,2.99,94,Clouds,broken clouds,0.89,2026-03-02T13:47:04.000Z
